# PHOEBE L1 Planck candidate — isolated Colab test

This notebook tests a narrow, **opt-in** candidate for the exact Roche-lobe L1 vertex found in the SZ Herculis semi-detached model. It does not modify PHOEBE Lab, the published app, or the installed PHOEBE package on disk.

The candidate retains PHOEBE's CK2004 atmosphere and existing CK2004→blackbody blending. Only a finite, positive blackbody temperature below the table floor is evaluated directly from the Planck function over the Johnson V passband. Non-positive, non-finite, and high out-of-range inputs retain PHOEBE's failure behavior. The temporary method is always restored.

Choose **Runtime → Run all** once. The final cell performs the complete test and prints one JSON report:

- reproduces the original five-phase NaN failure on the exact-axis L1 topology;
- computes SZ Her Preview (129 phases, 800 triangles) and Refined (513 phases, 1500 triangles);
- reports finite flux, passband luminosities, published-data residuals, and resolution convergence;
- enforces continuity where the direct quadrature meets the native blackbody table;
- proves ordinary detached and contact 129-phase outputs are bit-for-bit unchanged.

This is a scientific validation notebook, not the PHOEBE Lab fitting workbench. Do not use its candidate as an authoritative result until the hosted checks pass and the change is separately reviewed.

In [ ]:
#@title 1 · Install native PHOEBE
import importlib.metadata, os, subprocess, sys
os.environ.update(PHOEBE_ENABLE_ONLINE_PASSBANDS="FALSE", PHOEBE_ENABLE_MPI="FALSE",
                  PHOEBE_MULTIPROC_NPROCS="0", OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
try:
    installed = importlib.metadata.version("phoebe")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "2.5.4":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "phoebe==2.5.4", "scipy"])
import phoebe
assert phoebe.__version__ == "2.5.4", phoebe.__version__
print("Native PHOEBE", phoebe.__version__, "·", sys.executable)


In [ ]:
#@title 2 · Load the opt-in candidate
# Source embedded by the repository generator; no installed file is edited.
CANDIDATE_SOURCE = '"""Opt-in candidate for positive blackbody temperatures below the table floor.\n\nThis module is a validation artifact, not a production PHOEBE Lab patch. It\nleaves all atmosphere interpolation unchanged except where PHOEBE\'s blackbody\ntable returns a non-finite normal intensity for a finite, positive temperature\nbelow that table\'s lower bound. Those points are evaluated from the Planck\nfunction over the active passband with the requested photon/energy weighting.\n"""\nfrom contextlib import contextmanager\nfrom functools import wraps\n\nimport numpy as np\nfrom phoebe.atmospheres import models, passbands\n\n\ndef direct_blackbody_passband(passband, temperatures, weighting):\n    """Return un-limb-darkened normal intensities by direct passband quadrature."""\n    if weighting not in {"photon", "energy"}:\n        raise ValueError("weighting must be \'photon\' or \'energy\'")\n    temperatures = np.asarray(temperatures, dtype=float).reshape(-1)\n    wavelengths = np.asarray(passband.ptf_table["wl"], dtype=float)\n    transmission = np.asarray(passband.ptf_table["fl"], dtype=float)\n    area = passband.ptf_photon_area if weighting == "photon" else passband.ptf_area\n    answers = np.full(temperatures.shape, np.nan, dtype=float)\n    for index, temperature in enumerate(temperatures):\n        if not np.isfinite(temperature) or temperature <= 0:\n            continue\n        with np.errstate(over="ignore", divide="ignore", invalid="ignore", under="ignore"):\n            integrand = transmission * passband._planck(wavelengths, temperature)\n        if weighting == "photon":\n            integrand = wavelengths * integrand\n        answers[index] = np.trapezoid(integrand, wavelengths) / area\n    return answers\n\n\ndef _candidate_method(original):\n    @wraps(original)\n    def interpolate_inorms(\n        self,\n        query,\n        atm=models.CK2004ModelAtmosphere,\n        ldatm=models.CK2004ModelAtmosphere,\n        ldint=None,\n        ld_func="interp",\n        ld_coeffs=None,\n        intens_weighting="photon",\n        atm_extrapolation_method="none",\n        ld_extrapolation_method="none",\n        blending_method="none",\n        blending_margin=3,\n        dist_threshold=1e-5,\n    ):\n        result = original(\n            self,\n            query,\n            atm=atm,\n            ldatm=ldatm,\n            ldint=ldint,\n            ld_func=ld_func,\n            ld_coeffs=ld_coeffs,\n            intens_weighting=intens_weighting,\n            atm_extrapolation_method=atm_extrapolation_method,\n            ld_extrapolation_method=ld_extrapolation_method,\n            blending_method=blending_method,\n            blending_margin=blending_margin,\n            dist_threshold=dist_threshold,\n        )\n        if atm.name != "blackbody":\n            return result\n\n        temperatures = query.subset(atm.basic_axis_names).pts[:, 0]\n        values = np.asarray(result.interps, dtype=float)\n        use_direct = (\n            np.isfinite(temperatures)\n            & (temperatures > 0)\n            & (temperatures < atm.teffs[0])\n            & ~np.isfinite(values[:, 0])\n        )\n        if not np.any(use_direct):\n            return result\n\n        if ldint is None:\n            resolved_ldint = self.interpolate_ldints(\n                query=query,\n                ldatm=ldatm,\n                ld_func=ld_func,\n                ld_coeffs=ld_coeffs,\n                intens_weighting=intens_weighting,\n                ld_extrapolation_method=ld_extrapolation_method,\n            ).get_interpolated_values()\n        else:\n            resolved_ldint = ldint\n        resolved_ldint = np.asarray(resolved_ldint, dtype=float).reshape(-1)\n\n        values = values.copy()\n        values[use_direct, 0] = (\n            direct_blackbody_passband(self, temperatures[use_direct], intens_weighting)\n            / resolved_ldint[use_direct]\n        )\n        result.interps = values\n        return result\n\n    return interpolate_inorms\n\n\n@contextmanager\ndef low_temperature_planck_candidate():\n    """Temporarily enable the candidate and restore PHOEBE even after failure."""\n    original = passbands.Passband.interpolate_inorms\n    candidate = _candidate_method(original)\n    passbands.Passband.interpolate_inorms = candidate\n    try:\n        yield candidate\n    finally:\n        if passbands.Passband.interpolate_inorms is candidate:\n            passbands.Passband.interpolate_inorms = original\n'
exec(compile(CANDIDATE_SOURCE, "l1_planck_candidate.py", "exec"), globals())
import hashlib
print("Candidate SHA-256", hashlib.sha256(CANDIDATE_SOURCE.encode()).hexdigest())


In [ ]:
#@title 3 · Run complete native validation
VALIDATION_SOURCE = '"""Native end-to-end validation for the opt-in L1 Planck candidate."""\nfrom contextlib import contextmanager, nullcontext\nimport csv\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport os\nimport platform\nfrom pathlib import Path\nimport urllib.request\n\nos.environ.setdefault("PHOEBE_ENABLE_ONLINE_PASSBANDS", "FALSE")\nos.environ.setdefault("PHOEBE_ENABLE_MPI", "FALSE")\nos.environ.setdefault("PHOEBE_MULTIPROC_NPROCS", "0")\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\n\nimport numpy as np\nimport phoebe\nimport libphoebe\nfrom phoebe.atmospheres import models, passbands\nfrom phoebe.backend import universe\n\n\nBASE_REVISION = "fe9e3684eef2ba5ab7ab2aafee448a1f26abda98"\nSZ_HER_URL = (\n    "https://raw.githubusercontent.com/paulnord/phoebe-lab-preview/"\n    f"{BASE_REVISION}/site/examples/observed-sz-her-johnson-v.csv"\n)\nSZ_HER_SHA256 = "7cddf77b51037157490fcf4e9d68f8c5d0ef4a141ea44fd852f97a3fd47b1a91"\nEPOCH = 2454588.167997\nPERIOD = 0.8180837\n\n\ndef _sz_her_bundle(ntriangles, phases):\n    bundle = phoebe.default_binary(semidetached="secondary")\n    bundle.flip_constraint("mass@primary", solve_for="sma@binary")\n    bundle.set_value("mass", component="primary", value=1.2)\n    bundle.set_value("period", component="binary", value=PERIOD)\n    bundle.set_value("q", component="binary", value=0.4715)\n    sma = bundle.get_value("sma", component="binary", context="component")\n    bundle.set_value("requiv", component="primary", context="component", value=0.318 * sma)\n    bundle.set_value("teff", component="primary", value=6300.0)\n    bundle.set_value("teff", component="secondary", value=6300.0 * 4881.0 / 7262.0)\n    bundle.set_value("incl", component="binary", value=87.566)\n    bundle.set_value("irrad_method", value="none")\n    bundle.set_value_all("ntriangles", value=ntriangles)\n    bundle.add_dataset("lc", dataset="lc01", passband="Johnson:V", compute_phases=phases)\n    return bundle\n\n\ndef _control_bundle(geometry, phases):\n    bundle = phoebe.default_contact_binary() if geometry == "contact" else phoebe.default_binary()\n    bundle.set_value("irrad_method", value="none")\n    bundle.set_value_all("ntriangles", value=800)\n    bundle.add_dataset("lc", dataset="lc01", passband="Johnson:V", compute_phases=phases)\n    return bundle\n\n\n@contextmanager\ndef _exact_axis_secondary(enabled):\n    """Exercise the managed-Colab symmetry branch deterministically in local QA."""\n    if not enabled:\n        yield\n        return\n    original = universe.Star_roche.instantaneous_mesh_args\n\n    def exact_axis_args(self):\n        q, synchronicity, distance, spin, potential = original.fget(self)\n        if self.component == "secondary":\n            spin = np.array([0.0, 0.0, np.nextafter(1.0, 0.0)], dtype=np.float64)\n            target = np.float64(self.get_target_volume(scaled=False))\n            critical = np.float64(\n                libphoebe.roche_misaligned_critical_volume(q, synchronicity, distance, spin)\n            )\n            if target > critical and abs(target - critical) / target < 1e-10:\n                target = critical\n            potential = libphoebe.roche_misaligned_Omega_at_vol(\n                target, q, synchronicity, distance, spin\n            )\n        return q, synchronicity, distance, spin, potential\n\n    universe.Star_roche.instantaneous_mesh_args = property(exact_axis_args)\n    try:\n        yield\n    finally:\n        universe.Star_roche.instantaneous_mesh_args = original\n\n\ndef _luminosities(bundle):\n    raw = bundle.compute_pblums(\n        compute="phoebe01", pblum=True, pblum_abs=True,\n        pblum_scale=True, pbflux=True, set_value=False,\n    )\n    return {\n        str(key): float(np.asarray(getattr(value, "value", value), dtype=float).reshape(-1)[0])\n        for key, value in raw.items()\n    }\n\n\ndef _run(bundle, model, *, candidate, exact_axis=False):\n    candidate_context = low_temperature_planck_candidate() if candidate else nullcontext()\n    with _exact_axis_secondary(exact_axis), candidate_context:\n        luminosities = _luminosities(bundle)\n        bundle.run_compute(model=model, overwrite=True, progressbar=False)\n    flux = np.asarray(bundle.get_value("fluxes", dataset="lc01", model=model), dtype=float)\n    return flux, luminosities\n\n\ndef _load_observations(local_path=None):\n    if local_path is None:\n        payload = urllib.request.urlopen(SZ_HER_URL, timeout=30).read()\n    else:\n        payload = Path(local_path).read_bytes()\n    digest = hashlib.sha256(payload).hexdigest()\n    if digest != SZ_HER_SHA256:\n        raise RuntimeError(f"Unexpected SZ Her CSV SHA-256: {digest}")\n    rows = list(csv.DictReader(io.StringIO(payload.decode("utf-8"))))\n    times = np.asarray([float(row["hjd"]) for row in rows])\n    magnitudes = np.asarray([float(row["differential_johnson_v_magnitude"]) for row in rows])\n    phases = ((times - EPOCH) / PERIOD + 0.5) % 1.0 - 0.5\n    flux = 10.0 ** (-0.4 * (magnitudes - np.median(magnitudes)))\n    return phases, flux / np.median(flux), digest\n\n\ndef _residuals(phases, model, observed_phases, observed_flux):\n    normalized = model / np.median(model)\n    predicted = np.interp(observed_phases, phases, normalized)\n    residual = observed_flux - predicted\n    return {\n        "rms": float(np.sqrt(np.mean(residual**2))),\n        "median": float(np.median(residual)),\n        "max_abs": float(np.max(np.abs(residual))),\n        "count": int(residual.size),\n    }\n\n\ndef _native_hash():\n    origin = Path(importlib.util.find_spec("libphoebe").origin)\n    return hashlib.sha256(origin.read_bytes()).hexdigest()\n\n\ndef _blackbody_floor_continuity():\n    """Require continuity where the direct candidate meets the native table."""\n    pb = passbands.get_passband("Johnson:V")\n    floor = float(models.BlackbodyModelAtmosphere.teffs[0])\n    below = np.nextafter(floor, 0.0)\n    query = passbands.InterpQuery(\n        cols=["teffs"], pts=np.asarray([below, floor])[:, None]\n    )\n    results = {"temperature_floor_K": floor, "immediately_below_K": float(below)}\n    for weighting in ("photon", "energy"):\n        direct_at_floor = direct_blackbody_passband(pb, [floor], weighting)[0]\n        native_at_floor = pb.interpolate_inorms(\n            passbands.InterpQuery(cols=["teffs"], pts=np.asarray([[floor]])),\n            atm=models.BlackbodyModelAtmosphere,\n            ldatm=None,\n            ldint=np.ones(1),\n            ld_func="linear",\n            intens_weighting=weighting,\n        ).get_interpolated_values()[0, 0]\n        with low_temperature_planck_candidate():\n            patched = pb.interpolate_inorms(\n                query,\n                atm=models.BlackbodyModelAtmosphere,\n                ldatm=None,\n                ldint=np.ones(2),\n                ld_func="linear",\n                intens_weighting=weighting,\n            ).get_interpolated_values()[:, 0]\n        direct_vs_native = abs(direct_at_floor / native_at_floor - 1.0)\n        below_to_floor = abs(patched[0] / patched[1] - 1.0)\n        results[weighting] = {\n            "direct_vs_native_relative": float(direct_vs_native),\n            "below_to_floor_relative_jump": float(below_to_floor),\n        }\n        if direct_vs_native > 1e-10 or below_to_floor > 1e-10:\n            raise AssertionError(f"Blackbody {weighting} boundary is discontinuous")\n    return results\n\n\ndef run_validation(local_observations=None):\n    observed_phase, observed_flux, observation_sha = _load_observations(local_observations)\n    results = {\n        "status": "PASS",\n        "candidate": "opt-in-direct-low-temperature-planck",\n        "base_revision": BASE_REVISION,\n        "environment": {\n            "python": platform.python_version(),\n            "platform": platform.platform(),\n            "phoebe": phoebe.__version__,\n            "numpy": np.__version__,\n            "libphoebe_sha256": _native_hash(),\n            "observation_sha256": observation_sha,\n        },\n        "controls": {\n            "geometry": "semi-detached secondary donor",\n            "mass_primary_solar": 1.2,\n            "teff_primary_K": 6300.0,\n            "q_secondary_over_primary": 0.4715,\n            "inclination_deg": 87.566,\n            "primary_radius_over_sma": 0.318,\n            "temperature_ratio": 4881.0 / 7262.0,\n            "period_days": PERIOD,\n            "epoch_hjd": EPOCH,\n            "passband": "Johnson:V",\n            "irradiation": "none",\n            "atmosphere": "ck2004 with existing blackbody blending",\n        },\n        "sz_her": {},\n        "unchanged_controls": {},\n    }\n    results["blackbody_floor_continuity"] = _blackbody_floor_continuity()\n\n    original, _ = _run(\n        _sz_her_bundle(800, np.linspace(-0.5, 0.5, 5)),\n        "original_exact_axis", candidate=False, exact_axis=True,\n    )\n    results["original_exact_axis"] = {\n        "finite": int(np.isfinite(original).sum()), "samples": int(original.size)\n    }\n    if np.isfinite(original).any():\n        raise AssertionError("The deterministic exact-axis control did not reproduce the L1 failure")\n\n    natural, _ = _run(\n        _sz_her_bundle(800, np.linspace(-0.5, 0.5, 5)),\n        "natural_runtime_candidate", candidate=True, exact_axis=False,\n    )\n    results["natural_runtime_candidate"] = {\n        "finite": int(np.isfinite(natural).sum()), "samples": int(natural.size)\n    }\n    if not np.isfinite(natural).all():\n        raise AssertionError("The candidate did not produce a finite curve on this runtime\'s natural mesh")\n\n    curves = {}\n    for label, samples, triangles in (("Preview", 129, 800), ("Refined", 513, 1500)):\n        phases = np.linspace(-0.5, 0.5, samples)\n        flux, luminosities = _run(\n            _sz_her_bundle(triangles, phases),\n            label.lower(), candidate=True, exact_axis=True,\n        )\n        if flux.size != samples or not np.isfinite(flux).all() or np.any(flux <= 0):\n            raise AssertionError(f"{label} did not return {samples} finite positive samples")\n        curves[label] = {\n            "samples": samples,\n            "requested_ntriangles": triangles,\n            "finite": int(np.isfinite(flux).sum()),\n            "minimum": float(np.min(flux)),\n            "maximum": float(np.max(flux)),\n            "median": float(np.median(flux)),\n            "luminosities": luminosities,\n            "residuals": _residuals(phases, flux, observed_phase, observed_flux),\n            "phases": phases,\n            "flux": flux,\n        }\n    preview = curves["Preview"]\n    refined = curves["Refined"]\n    refined_at_preview = np.interp(preview["phases"], refined["phases"], refined["flux"])\n    difference = preview["flux"] / np.median(preview["flux"]) - refined_at_preview / np.median(refined_at_preview)\n    results["resolution_convergence"] = {\n        "normalized_rms": float(np.sqrt(np.mean(difference**2))),\n        "normalized_max_abs": float(np.max(np.abs(difference))),\n        "secondary_pblum_abs_ratio_preview_over_refined": (\n            preview["luminosities"]["pblum_abs@secondary@lc01"]\n            / refined["luminosities"]["pblum_abs@secondary@lc01"]\n        ),\n    }\n    for label, curve in curves.items():\n        results["sz_her"][label] = {\n            key: value for key, value in curve.items() if key not in {"phases", "flux"}\n        }\n\n    control_phases = np.linspace(-0.5, 0.5, 129)\n    for geometry in ("detached", "contact"):\n        baseline, _ = _run(_control_bundle(geometry, control_phases), f"{geometry}_baseline", candidate=False)\n        candidate, _ = _run(_control_bundle(geometry, control_phases), f"{geometry}_candidate", candidate=True)\n        equal = bool(np.array_equal(baseline, candidate))\n        results["unchanged_controls"][geometry] = {\n            "samples": 129,\n            "array_equal": equal,\n            "max_abs_difference": float(np.max(np.abs(baseline - candidate))),\n        }\n        if not equal:\n            raise AssertionError(f"Candidate changed the ordinary {geometry} control")\n    return results\n\n\ndef printable_results(results):\n    return json.dumps(results, indent=2, sort_keys=True)\n\n'
exec(compile(VALIDATION_SOURCE, "l1_planck_validation.py", "exec"), globals())
validation_results = run_validation()
print(printable_results(validation_results))
assert validation_results["status"] == "PASS"


## Reading the result

`status: PASS` means the candidate completed the exact-axis Preview and Refined runs, returned finite positive native curves, computed finite luminosities and residuals, and left the detached/contact controls exactly unchanged in this runtime.

The residuals compare the fixed PHOEBE Lab assumptions to published SZ Her Johnson V photometry. They are a regression check, not a new scientific fit and not evidence that the parameters are unique. The notebook downloads that unchanged CSV from immutable source revision `fe9e3684eef2ba5ab7ab2aafee448a1f26abda98` and verifies SHA-256 before use.

If any assertion fails, copy the final cell output together with the environment and native-library hashes printed in the report. No app or package state remains patched after the cell finishes.